In [1]:
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
for folder in [RAW_DIR, PROCESSED_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Download IndicXNLI (Hindi & Telugu)
from huggingface_hub import hf_hub_download
REPO_ID = "Divyanshu/indicxnli"
FILES = [
    "forward/train/xnli_hi.json", "forward/dev/xnli_hi.json", "forward/test/xnli_hi.json",
    "forward/train/xnli_te.json", "forward/dev/xnli_te.json", "forward/test/xnli_te.json",
]
for file in FILES:
    path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=file,
                            local_dir=RAW_DIR, local_dir_use_symlinks=False)
    print(path)

# JSON loader
import json, pandas as pd
def load_xnli(language: str, split: str) -> pd.DataFrame:
    path = RAW_DIR / "forward" / split / f"xnli_{language}.json"
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    records = next(iter(data.values()))
    df = pd.DataFrame(records)
    df["language"] = language
    df["split"] = split
    return df

hi_train, hi_dev, hi_test = load_xnli("hi","train"), load_xnli("hi","dev"), load_xnli("hi","test")
te_train, te_dev, te_test = load_xnli("te","train"), load_xnli("te","dev"), load_xnli("te","test")

# Profiler
def profile_dataset(df, name):
    print("="*80); print(name); print("="*80)
    print(f"Shape: {df.shape}")
    print(f"Missing:\n{df.isnull().sum()}")
    print(f"Duplicates: {df.duplicated(subset=['premise','hypothesis','label']).sum()}")
    print(f"Label dist:\n{df['label'].value_counts().sort_index()}")

profile_dataset(hi_train, "Hindi Train")
profile_dataset(te_train, "Telugu Train")

# Label distribution plots, sentence length plots, split-overlap check
def check_split_overlap(df1, df2, name1, name2):
    cols = ["premise","hypothesis","label"]
    overlap = pd.merge(df1[cols], df2[cols], on=cols, how="inner").drop_duplicates()
    print(f"{name1} ↔ {name2} Overlap: {len(overlap)}")

check_split_overlap(hi_train, hi_dev, "HI Train", "HI Dev")
check_split_overlap(hi_train, hi_test, "HI Train", "HI Test")
check_split_overlap(te_train, te_dev, "TE Train", "TE Dev")
check_split_overlap(te_train, te_test, "TE Train", "TE Test")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


forward/train/xnli_hi.json:   0%|          | 0.00/219M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/train/xnli_hi.json


forward/dev/xnli_hi.json:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/dev/xnli_hi.json


...

/kaggle/working/data/raw/forward/train/xnli_te.json


forward/dev/xnli_te.json:   0%|          | 0.00/1.33M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/dev/xnli_te.json


forward/test/xnli_te.json:   0%|          | 0.00/2.70M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/test/xnli_te.json
Hindi Train
Shape: (392702, 5)
Missing:
premise       0
hypothesis    0
label         0
language      0
split         0
dtype: int64
Duplicates: 77
Label dist:
label
0    130899
1    130900
2    130903
Name: count, dtype: int64
Telugu Train
Shape: (392702, 5)
Missing:
premise       0
hypothesis    0
label         0
language      0
split         0
dtype: int64
Duplicates: 73
Label dist:
label
0    130899
1    130900
2    130903
Name: count, dtype: int64
HI Train ↔ HI Dev Overlap: 0
HI Train ↔ HI Test Overlap: 0
TE Train ↔ TE Dev Overlap: 0
TE Train ↔ TE Test Overlap: 0
